# Aula 19 — Model Context Protocol (MCP)

Na Aula 17, estruturamos **tools**. Na Aula 18, coordenamos tools em **workflows determinísticos**.

Agora avançamos para uma nova pergunta:

> **Como expor e descobrir capacidades de forma padronizada sem confundir protocolo de integração com autonomia agente?**

```text
Aula 17 → contrato de capacidade
Aula 18 → execução governada
Aula 19 → exposição + descoberta padronizadas
```


## Objetivos

Ao final da aula, você deverá conseguir:

- explicar qual problema de integração o MCP resolve;
- distinguir **host**, **client** e **server**;
- distinguir **tool**, **resource** e **prompt**;
- observar protocol version, server info e capabilities;
- executar discovery e invocation;
- compreender a diferença entre capability semantics e transport;
- relacionar MCP a autorização, observabilidade e governança;
- justificar quando MCP adiciona utility e quando uma integração direta é suficiente.


## Glossário da aula

Esta aula introduz novos termos que serão incorporados ao **Glossário Vivo**:

**Model Context Protocol · MCP Host · MCP Client · MCP Server · Capability Discovery · MCP Tool · MCP Resource · MCP Prompt · Protocol Version · Transport · Authorization**

Glossário PT-BR:  
https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md

> Durante a revisão final, os termos receberão links contextuais individuais antes da promoção para `student-ready`.


## 1. Antes do MCP: qual é o problema?

Imagine três aplicações que precisam acessar o mesmo conjunto de capacidades:

```text
App A ── integração própria ── Serviço X
App B ── integração própria ── Serviço X
App C ── integração própria ── Serviço X
```

Cada integração pode repetir descoberta, schemas, convenções de chamada, tratamento de erros e segurança.

A pergunta é:

> O problema aqui é inteligência do modelo ou **padronização da integração**?

```text
integration problem
≠
reasoning problem
```


## 2. O que MCP adiciona?

```text
AI Application / Host
        ↓
MCP Client
        ↓
standardized protocol
        ↓
MCP Server
        ↓
tools / resources / prompts
```

O MCP padroniza como capacidades e contexto podem ser **expostos, descobertos e invocados**.


## 3. MCP não é agente

```text
MCP
→ protocolo de integração

workflow
→ sequência e política de execução

agent
→ sistema com algum grau de autonomia para decidir e agir
```

Portanto:

```text
MCP ≠ workflow
MCP ≠ agent
MCP ≠ authorization policy
```

O protocolo não substitui os contratos da Aula 17 nem a governança da Aula 18.


## 4. Disciplina de versão

Esta aula usa explicitamente:

```text
MCP specification = 2026-07-28
Python SDK         = 2.x
```

Protocolos evoluem. Um tutorial sem versão pode ensinar corretamente uma revisão antiga e, ao mesmo tempo, induzir erro sobre o comportamento atual.

> **Protocolos versionados devem ser estudados junto com a versão da especificação.**


## 5. Ambiente

O notebook não instala pacotes durante a execução.

Ele falha cedo se o SDK MCP 2.x não estiver presente. Isso preserva a reprodutibilidade do ambiente oficial.


In [ ]:
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import subprocess
import sys

REQUIRED_MCP_VERSION = "2.2.0"
KAGGLE_WHEELHOUSE = Path("/kaggle/input/til-mcp-python-sdk-wheelhouse")

def ensure_mcp():
    try:
        installed = version("mcp")
        print("MCP já disponível:", installed)
        return installed
    except PackageNotFoundError:
        pass

    if KAGGLE_WHEELHOUSE.exists():
        print("MCP não encontrado no ambiente.")
        print("Instalando offline a partir do Kaggle Dataset:", KAGGLE_WHEELHOUSE)

        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--no-index",
                "--find-links",
                str(KAGGLE_WHEELHOUSE),
                f"mcp=={REQUIRED_MCP_VERSION}",
            ],
            check=True,
        )
        return version("mcp")

    raise RuntimeError(
        "O pacote 'mcp' não está disponível e o wheelhouse offline não foi encontrado. "
        "No Kaggle, anexe o dataset 'pedrogentil/til-mcp-python-sdk-wheelhouse'. "
        "Localmente, instale previamente o MCP SDK 2.x."
    )

MCP_SDK_VERSION = ensure_mcp()

major = int(MCP_SDK_VERSION.split(".", 1)[0])
if major < 2:
    raise RuntimeError(
        f"MCP SDK incompatível: {MCP_SDK_VERSION}. "
        "A Aula 19 requer a linha 2.x."
    )

print("MCP Python SDK:", MCP_SDK_VERSION)
print("Baseline do protocolo: 2026-07-28")


In [ ]:
from time import perf_counter
import json
import pandas as pd

from mcp import Client
from mcp.server import MCPServer
from mcp.types import TextContent, TextResourceContents

print("Imports carregados.")


## 6. Dados simulados do laboratório

Assim como na Aula 18, usaremos um pequeno estado em memória.

> Estes dados existem apenas para fins didáticos e **não representam o estado real do TIL**.


In [ ]:
SIMULATED_LESSON_STATUS = {
    "17": "Available",
    "18": "Available",
    "19": "Draft",
}

SIMULATED_LESSON_STATUS


## 7. Criando o MCP Server

No SDK de alto nível, funções Python podem ser registradas como capabilities:

```text
Python function
      ↓
decorator MCP
      ↓
registered capability
      ↓
MCPServer
```

O servidor não será iniciado em uma porta. Usaremos uma conexão **in-process**.


In [ ]:
mcp = MCPServer(
    "TIL Lesson Server",
    instructions=(
        "Servidor didático da Aula 19. "
        "Expõe apenas capacidades locais e sem side effects reais."
    ),
)

print("MCPServer criado.")


## 8. Resource

Uma distinção fundamental:

```text
resource
→ conteúdo que a aplicação lê

tool
→ operação que pode ser chamada
```

Resources possuem um **endereço**, normalmente uma URI.


In [ ]:
@mcp.resource(
    "til://lessons/status",
    title="TIL simulated lesson status",
    description="Estado didático e simulado das aulas 17, 18 e 19.",
    mime_type="application/json",
)
def lesson_status_resource() -> str:
    """Return the simulated TIL lesson status table."""
    return json.dumps(SIMULATED_LESSON_STATUS, ensure_ascii=False, indent=2)

print("Resource registrado: til://lessons/status")


## 9. Tool — `get_lesson_status`

A tool mantém a disciplina aprendida na Aula 17:

```text
name
+ description
+ typed input
+ validation
+ result
```

O SDK deriva o input schema das anotações de tipo da função.


In [ ]:
@mcp.tool(
    title="Get lesson status",
    description="Return the simulated status of one TIL lesson.",
)
def get_lesson_status(lesson_id: str) -> dict:
    """Return the simulated status of one lesson."""
    if lesson_id not in SIMULATED_LESSON_STATUS:
        raise ValueError(f"lesson_not_found:{lesson_id}")

    return {
        "lesson_id": lesson_id,
        "status": SIMULATED_LESSON_STATUS[lesson_id],
    }


@mcp.tool(
    title="Build release note",
    description="Build a release note without producing external side effects.",
)
def build_release_note(lesson_id: str, status: str) -> dict:
    """Build a deterministic release note."""
    return {
        "lesson_id": lesson_id,
        "release_note": f"Aula {lesson_id}: status atual = {status}.",
    }

print("Tools registradas.")


## 10. Prompt — `review_release`

```text
prompt
→ template que host/usuário pode selecionar e renderizar

prompt ≠ tool
prompt ≠ security policy
```

Neste laboratório, o prompt é apenas renderizado. Nenhum LLM externo será chamado.


In [ ]:
@mcp.prompt(
    title="Review release",
    description="Prepare a release note for human review.",
)
def review_release(release_note: str) -> str:
    """Create a human-review instruction for a release note."""
    return (
        "Revise a nota abaixo antes de qualquer publicação:\n\n"
        f"{release_note}\n\n"
        "Verifique clareza, status e ausência de efeitos externos não autorizados."
    )

print("Prompt registrado: review_release")


## 11. O cliente MCP

Usaremos:

```python
Client(mcp)
```

Isso conecta cliente e servidor no mesmo processo:

```text
Client
  ↕
DirectDispatcher
  ↕
MCPServer
```

Não há subprocesso, porta ou framing de rede nesta conexão didática.

A vantagem é isolar os **conceitos do protocolo e das capabilities** antes de introduzir transporte.


## 12. Protocol Introspection Lab

Antes de chamar qualquer capability, vamos observar:

```text
protocol_version
server_info
server_capabilities
instructions
```

A pergunta é:

> **O que o cliente já consegue saber antes de executar uma tool?**


In [ ]:
async with Client(mcp) as client:
    protocol_snapshot = {
        "protocol_version": client.protocol_version,
        "server_name": client.server_info.name if client.server_info else None,
        "server_version": client.server_info.version if client.server_info else None,
        "has_tools": client.server_capabilities.tools is not None,
        "has_resources": client.server_capabilities.resources is not None,
        "has_prompts": client.server_capabilities.prompts is not None,
        "instructions": client.instructions,
    }

pd.DataFrame([protocol_snapshot]).T.rename(columns={0: "value"})


## 13. Discovery

Discovery permite que o cliente inspecione as capabilities e seus contratos sem conhecer o código Python interno do servidor.

Vamos começar pelas tools.


In [ ]:
async with Client(mcp) as client:
    tools_result = await client.list_tools()

tools_view = pd.DataFrame([
    {
        "name": tool.name,
        "title": tool.title,
        "description": tool.description,
        "input_schema": tool.input_schema,
    }
    for tool in tools_result.tools
])

display(tools_view)


In [ ]:
async with Client(mcp) as client:
    resources_result = await client.list_resources()
    templates_result = await client.list_resource_templates()
    prompts_result = await client.list_prompts()

resources_view = pd.DataFrame([
    {"uri": str(resource.uri), "name": resource.name, "title": resource.title}
    for resource in resources_result.resources
])

templates_view = pd.DataFrame([
    {"uri_template": template.uri_template, "name": template.name, "title": template.title}
    for template in templates_result.resource_templates
])

prompts_view = pd.DataFrame([
    {
        "name": prompt.name,
        "title": prompt.title,
        "arguments": [arg.name for arg in (prompt.arguments or [])],
    }
    for prompt in prompts_result.prompts
])

print("Resources")
display(resources_view)
print("Resource templates")
display(templates_view)
print("Prompts")
display(prompts_view)


## 14. Invocando uma tool

Agora o fluxo deixa de ser apenas discovery:

```text
discover
→ select capability
→ invoke
→ inspect result
```


In [ ]:
async with Client(mcp) as client:
    tool_result = await client.call_tool(
        "get_lesson_status",
        {"lesson_id": "18"},
    )

print("is_error:", tool_result.is_error)
print("structured_content:", tool_result.structured_content)

for block in tool_result.content:
    if isinstance(block, TextContent):
        print("text:", block.text)


## 15. Lendo um resource

Resources são endereçados por URI:

```text
resource address
→ URI
→ read_resource(...)
```


In [ ]:
async with Client(mcp) as client:
    resource_result = await client.read_resource("til://lessons/status")

for item in resource_result.contents:
    if isinstance(item, TextResourceContents):
        print(item.text)


## 16. Renderizando um prompt

O MCP entrega mensagens estruturadas ao host.

Ele **não precisa executar um modelo** para que o mecanismo de prompt seja útil ou observável.


In [ ]:
async with Client(mcp) as client:
    prompt_result = await client.get_prompt(
        "review_release",
        {"release_note": "Aula 19: Draft."},
    )

for message in prompt_result.messages:
    content = message.content
    text = content.text if isinstance(content, TextContent) else str(content)
    print("role:", message.role)
    print("content:", text)


## 17. Três primitives, três papéis

| Primitive | Papel didático | Quem decide usar? |
|---|---|---|
| **Tool** | operação | modelo/host sob política |
| **Resource** | conteúdo/contexto | aplicação/host |
| **Prompt** | template selecionável | usuário/host |

Uma diferença de protocolo importante é também uma diferença de responsabilidade.


## 18. Discovery vs integração hard-coded

```text
hard-coded integration
→ cliente conhece previamente endpoint/assinatura

MCP discovery
→ cliente pode inspecionar capabilities e contratos
```

Discovery não elimina integração, validação, autorização ou governança. Ele **padroniza uma parte do problema**.


## 19. Protocol version e compatibilidade

O cliente moderno usa `mode="auto"` por padrão.

Conceitualmente:

```text
modern server
→ server/discover
→ 2026-07-28

older server
→ fallback para handshake legado
```

Neste laboratório, `Client(mcp)` conversa com o nosso próprio `MCPServer` 2.x e chega ao caminho moderno.


In [ ]:
async with Client(mcp) as client:
    print("protocol_version:", client.protocol_version)
    print("server_info:", client.server_info)


## 20. Transportes

As mesmas capabilities podem ser expostas por transportes diferentes:

```text
                 ┌→ in-process
capabilities ────┼→ stdio
                 └→ Streamable HTTP
```

```text
capability semantics
≠
transport
```

Nesta aula, o caminho obrigatório é **in-process**. stdio e Streamable HTTP ficam como extensão conceitual.


## 21. Segurança e autorização

Duas separações importantes:

```text
model confidence
≠
authorization

capability exists
≠
caller is authorized
```

A existência de uma tool não concede automaticamente permissão para executá-la.

Princípios:

- menor privilégio;
- secrets fora de prompts;
- validação de argumentos;
- autorização fora da decisão probabilística do modelo;
- approval gates para ações relevantes;
- auditabilidade.


## 22. Multi Round-Trip Requests (MRTR)

Na revisão 2026-07-28, quando uma operação precisa de informação adicional, o fluxo moderno pode ser:

```text
client request
→ server retorna input_required
→ client obtém a informação
→ repete a request original com inputResponses
→ server continua
```

Nesta primeira aula, o mecanismo é apenas conceitual. Não precisamos implementá-lo para compreender sua função.


## 23. O que mudou em 2026

Materiais antigos podem mencionar padrões que não representam o caminho moderno:

```text
sampling               → deprecated
roots                  → deprecated
initialize/initialized → caminho legado
Mcp-Session-Id         → ausente do caminho moderno 2026
```

O objetivo não é descartar documentação histórica, mas **ler cada material dentro da versão que ele ensina**.


## 24. Observabilidade

Vamos registrar cada operação MCP usada no laboratório.

Campos mínimos:

```text
protocol_version
operation
capability
latency_ms
status
error_type
```

O protocolo não elimina observabilidade; ele cria novas superfícies que também precisam ser observadas.


In [ ]:
MCP_EVENTS = []

async def observed_call(operation: str, capability: str, fn):
    start = perf_counter()
    event = {
        "protocol_version": None,
        "operation": operation,
        "capability": capability,
        "status": "running",
        "error_type": None,
        "latency_ms": None,
    }

    try:
        async with Client(mcp) as client:
            event["protocol_version"] = client.protocol_version
            result = await fn(client)
            event["status"] = "ok"
            return result
    except Exception as exc:
        event["status"] = "failed"
        event["error_type"] = type(exc).__name__
        raise
    finally:
        event["latency_ms"] = (perf_counter() - start) * 1000
        MCP_EVENTS.append(event)


status_result = await observed_call(
    "tools/call",
    "get_lesson_status",
    lambda client: client.call_tool(
        "get_lesson_status",
        {"lesson_id": "18"},
    ),
)

resource_observed = await observed_call(
    "resources/read",
    "til://lessons/status",
    lambda client: client.read_resource("til://lessons/status"),
)

display(pd.DataFrame(MCP_EVENTS))


## 25. Failure Lab

Vamos distinguir falhas diferentes:

| Caso | Exemplo |
|---|---|
| capability not found | tool inexistente |
| server execution failure | `lesson_id` inexistente |
| invalid arguments | argumento obrigatório ausente |
| resource not found | URI inexistente |

Dizer apenas **"MCP falhou"** seria tão insuficiente quanto dizer apenas **"workflow falhou"**.


In [ ]:
# Exercício
#
# Execute pelo menos dois casos de falha e observe:
# - exceção ou CallToolResult;
# - is_error;
# - conteúdo retornado;
# - tipo de falha.
#
# Sugestões:
# await client.call_tool("missing_tool", {})
# await client.call_tool("get_lesson_status", {"lesson_id": "999"})
# await client.call_tool("get_lesson_status", {})
# await client.read_resource("til://missing")


### Dica

Uma tool que lança exceção pode ser representada como um `CallToolResult` com `is_error=True`.

Outras falhas de protocolo podem chegar como exceções.

Portanto, observe **o contrato do resultado** antes de assumir que todo erro terá a mesma forma.


In [ ]:
failure_rows = []

async with Client(mcp) as client:
    cases = [
        ("missing_tool", lambda: client.call_tool("missing_tool", {})),
        (
            "lesson_not_found",
            lambda: client.call_tool(
                "get_lesson_status",
                {"lesson_id": "999"},
            ),
        ),
        (
            "missing_argument",
            lambda: client.call_tool("get_lesson_status", {}),
        ),
        (
            "missing_resource",
            lambda: client.read_resource("til://missing"),
        ),
    ]

    for name, action in cases:
        try:
            result = await action()
            failure_rows.append({
                "case": name,
                "result_type": type(result).__name__,
                "is_error": getattr(result, "is_error", None),
                "exception_type": None,
            })
        except Exception as exc:
            failure_rows.append({
                "case": name,
                "result_type": None,
                "is_error": None,
                "exception_type": type(exc).__name__,
            })

display(pd.DataFrame(failure_rows))


## 26. Architecture Decision Lab

Compare:

```text
A — direct_function
B — structured_tool
C — mcp_capability
```

Critérios:

- quantos consumidores existem?
- discovery é necessário?
- interoperabilidade importa?
- há necessidade de isolamento?
- quais controles de segurança existem?
- como a execução será observada?
- qual é o custo operacional adicional?

> **Escolha a arquitetura mínima suficiente.**


In [ ]:
architecture_cases = pd.DataFrame([
    {
        "case": "A",
        "situation": "Uma função é usada apenas dentro do mesmo módulo Python.",
        "multiple_consumers": False,
        "discovery_needed": False,
    },
    {
        "case": "B",
        "situation": "Um workflow local chama uma operação com schema e validação explícitos.",
        "multiple_consumers": False,
        "discovery_needed": False,
    },
    {
        "case": "C",
        "situation": "Hosts diferentes precisam descobrir e usar o mesmo conjunto de capabilities.",
        "multiple_consumers": True,
        "discovery_needed": True,
    },
    {
        "case": "D",
        "situation": "Uma aplicação simples chama uma única função interna e nunca terá consumidores externos.",
        "multiple_consumers": False,
        "discovery_needed": False,
    },
])

display(architecture_cases)


### Sua tarefa

Escolha para cada caso:

- `direct_function`
- `structured_tool`
- `mcp_capability`

e escreva uma justificativa curta.

Não escolha MCP apenas porque ele é mais novo.


In [ ]:
architecture_reference = {
    "A": {
        "choice": "direct_function",
        "reason": "Não há necessidade de discovery ou interoperabilidade.",
    },
    "B": {
        "choice": "structured_tool",
        "reason": "O contrato ajuda, mas a exposição por protocolo ainda não compra utility suficiente.",
    },
    "C": {
        "choice": "mcp_capability",
        "reason": "Múltiplos hosts e discovery tornam a padronização relevante.",
    },
    "D": {
        "choice": "direct_function",
        "reason": "Adicionar uma camada de protocolo seria complexidade sem benefício observável.",
    },
}

reference_view = architecture_cases.copy()
reference_view["reference_choice"] = reference_view["case"].map(
    lambda case: architecture_reference[case]["choice"]
)
reference_view["reason"] = reference_view["case"].map(
    lambda case: architecture_reference[case]["reason"]
)

display(reference_view)


## 27. Síntese

A progressão do Bloco C agora é:

```text
Aula 17
capability contract
      ↓
Aula 18
controlled execution
      ↓
Aula 19
standardized exposure + discovery
```

Um MCP Server pode expor tools, resources e prompts.

Um MCP Client pode descobrir e invocar essas capabilities.

Mas:

> **MCP resolve um problema de integração. Ele não cria autonomia por si só.**


## 28. Ponte para Agentic Systems

MCP pode se tornar parte de um sistema agente:

```text
MCP capabilities
+ workflow / execution policy
+ model decisions
+ state
+ human oversight
        ↓
possible agentic system
```

Mas a relação correta não é:

```text
MCP → agent
```

A próxima etapa do TIL deverá estudar **autonomia** como propriedade separada e mensurável.


## 29. Reprodutibilidade

O núcleo da Aula 19 foi desenhado para:

```text
Internet OFF
GPU OFF
MCP client/server in-process
MCP 2.2.0 via Kaggle Dataset offline
sem API proprietária
sem side effects externos
sem servidor persistente
sem porta local
```

O SDK e a versão do protocolo são explicitados porque fazem parte do contrato de execução.


---

## Continue no TIL

← **[Anterior: Aula 18 — Deterministic Workflows](https://www.kaggle.com/code/pedrogentil/til-18-deterministic-workflows)** &nbsp;&nbsp;|&nbsp;&nbsp; 🏠 **[Apresentação do curso](https://www.kaggle.com/code/pedrogentil/text-intelligence-lab-course)** &nbsp;&nbsp;|&nbsp;&nbsp; **Próxima unidade: Agentic Systems — em preparação** →
